# Backend Tester — MTS AI Hub
Тестирует все эндпоинты бэкенда на `http://localhost:8000`

In [4]:
import httpx
import json
import base64
import asyncio
from pathlib import Path

BASE = "http://localhost:8000"
HEADERS = {"Content-Type": "application/json"}

def ok(label, resp):
    status = '✅' if resp.status_code < 400 else '❌'
    print(f"{status} [{resp.status_code}] {label}")
    try:
        data = resp.json()
        print(json.dumps(data, ensure_ascii=False, indent=2)[:800])
    except Exception:
        print(resp.text[:400])
    print()

## 1. Health Check

In [5]:
r = httpx.get(f"{BASE}/v1/health")
ok("Health Check", r)

✅ [200] Health Check
{
  "status": "degraded",
  "services": {
    "postgres": true,
    "mws_api": true,
    "ollama": false,
    "asr": false,
    "image_gen": false,
    "vlm": false
  }
}



## 2. Chat — Text (обычный вопрос)

In [19]:
payload = {
    "model": "gemma-3-27b-it",
    "messages": [{"role": "user", "content": "Привет! Кто ты?"}],
    "stream": False,
    "user": "test_user"
}
r = httpx.post(f"{BASE}/v1/chat/completions", json=payload, timeout=30)
ok("Chat / Text", r)

✅ [200] Chat / Text
{
  "id": "chatcmpl-b78438571fb35a3c",
  "created": 1775906624,
  "model": "mws-gpt-alpha",
  "object": "chat.completion",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "Привет! Я - Модель Высшего Уровня Знаний (МВС GPT) - большая языковая модель, основанная на архитектуре GPT. Я создана для ответа на вопросы и помогать людям найти информацию и решение для различных задач. Моя основная задача - предоставлять точные, ясные, полезные и содержательные ответы на ваши вопросы. Я всегда готова помочь и ответить на любые вопросы, которые вас интересуют!",
        "role": "assistant"
      }
    }
  ],
  "usage": {
    "completion_tokens": 117,
    "prompt_tokens": 299,
    "total_tokens": 416
  }
}



## 3. Chat — Code (роутинг на kodify-2.0)

In [26]:
payload = {
    "model": "qwen3-coder-480b-a35b",
    "messages": [{"role": "user", "content": "Напиши функцию сортировки пузырьком на Python"}],
    "stream": False,
    "user": "test_user"
}
r = httpx.post(f"{BASE}/v1/chat/completions", json=payload, timeout=30)
ok("Chat / Code routing", r)

❌ [500] Chat / Code routing
Internal Server Error



## 4. Chat — Streaming (SSE)

In [ ]:
payload = {
    "model": "mws-gpt-alpha",
    "messages": [{"role": "user", "content": "Расскажи анекдот"}],
    "stream": True,
    "user": "test_user"
}
print("🔄 Streaming response:")
chunks = []
with httpx.stream("POST", f"{BASE}/v1/chat/completions", json=payload, timeout=30) as resp:
    print(f"Status: {resp.status_code}")
    for line in resp.iter_lines():
        if line.startswith("data: ") and line != "data: [DONE]":
            try:
                chunk = json.loads(line[6:])
                delta = chunk["choices"][0]["delta"].get("content", "")
                if delta:
                    chunks.append(delta)
                    print(delta, end="", flush=True)
            except Exception:
                pass
print(f"\n\n✅ Total chunks: {len(chunks)}")

## 5. Deep Research

In [ ]:
payload = {
    "query": "Что такое квантовые вычисления?",
    "user_id": "test_user"
}
print("🔄 Research streaming:")
with httpx.stream("POST", f"{BASE}/v1/research", json=payload, timeout=60) as resp:
    print(f"Status: {resp.status_code}")
    lines_collected = []
    for line in resp.iter_lines():
        if line:
            lines_collected.append(line)
            print(line[:120])
        if len(lines_collected) > 10:
            print("... (truncated)")
            break
print("\n✅ Research streaming OK")

## 6. Web Search

In [ ]:
payload = {"query": "Python FastAPI tutorial", "max_results": 3}
r = httpx.post(f"{BASE}/v1/tools/web-search", json=payload, timeout=20)
ok("Web Search", r)

## 7. Web Parse

In [ ]:
payload = {"url": "https://example.com"}
r = httpx.post(f"{BASE}/v1/tools/web-parse", json=payload, timeout=20)
ok("Web Parse", r)

## 8. File Upload (TXT)

In [ ]:
# Create a temp file to upload
test_txt = Path("/tmp/test_doc.txt")
test_txt.write_text("Это тестовый документ. Python — мощный язык программирования. FastAPI очень быстрый.", encoding="utf-8")

with open(test_txt, "rb") as f:
    r = httpx.post(
        f"{BASE}/v1/files/upload",
        files={"file": ("test_doc.txt", f, "text/plain")},
        data={"user_id": "test_user"},
        timeout=30
    )
ok("File Upload (TXT)", r)

## 9. Memory — Get User Memory

In [ ]:
r = httpx.get(f"{BASE}/v1/memory/test_user", timeout=10)
ok("Memory GET test_user", r)

## 10. Memory — Search

In [ ]:
payload = {"query": "Python программирование", "top_k": 5}
r = httpx.post(f"{BASE}/v1/memory/test_user/search", json=payload, timeout=10)
ok("Memory Search", r)

## 11. Image Generation

In [ ]:
payload = {"prompt": "beautiful sunset over the sea", "width": 512, "height": 512}
r = httpx.post(f"{BASE}/v1/image/generate", json=payload, timeout=60)
ok("Image Generate", r)

## 12. PPTX Generation

In [ ]:
payload = {
    "topic": "Искусственный интеллект в 2025 году",
    "slides_count": 5,
    "user_id": "test_user"
}
r = httpx.post(f"{BASE}/v1/tools/generate-pptx", json=payload, timeout=60)
if r.status_code == 200:
    out = Path("/tmp/test_output.pptx")
    out.write_bytes(r.content)
    print(f"✅ [200] PPTX saved to {out} ({len(r.content)} bytes)")
else:
    ok("PPTX Generate", r)

## 13. Voice Message (ASR → LLM → TTS)

In [ ]:
# Requires a real WAV file; here we test the endpoint exists and returns 4xx without audio
r = httpx.post(
    f"{BASE}/v1/voice/message",
    files={"audio": ("test.wav", b"RIFF", "audio/wav")},
    data={"user_id": "test_user"},
    timeout=30
)
status = '✅ endpoint reachable' if r.status_code < 500 else '❌ server error'
print(f"{status} [{r.status_code}] Voice Message")
print(r.text[:300])

## 14. Router Test (детектирование типов)

In [ ]:
test_cases = [
    ("Привет, как дела?",                       "text"),
    ("Напиши функцию сортировки на Python",      "code"),
    ("Исследуй тему квантовых вычислений",       "deep_research"),
    ("Зайди на https://example.com и расскажи",  "web_parse"),
    ("Найди новости про ИИ",                     "web_search"),
    ("Нарисуй закат над морем",                  "image_gen"),
]

print("Testing router via /v1/chat/completions with model=auto\n")
for msg, expected in test_cases:
    payload = {
        "model": "auto",
        "messages": [{"role": "user", "content": msg}],
        "stream": False,
        "user": "test_user"
    }
    try:
        r = httpx.post(f"{BASE}/v1/chat/completions", json=payload, timeout=15)
        routed_model = r.json().get("model", "?")
        status = "✅" if r.status_code < 400 else "❌"
        print(f"{status} [{expected:15s}] model={routed_model:30s} | {msg[:50]}")
    except Exception as e:
        print(f"❌ [{expected:15s}] ERROR: {e} | {msg[:50]}")

## 15. Summary — All Endpoints

In [ ]:
endpoints = [
    ("GET",  "/v1/health",                    None),
    ("POST", "/v1/chat/completions",          {"model":"mws-gpt-alpha","messages":[{"role":"user","content":"hi"}],"stream":False}),
    ("POST", "/v1/research",                  {"query":"test","user_id":"u1"}),
    ("POST", "/v1/tools/web-search",          {"query":"test","max_results":1}),
    ("POST", "/v1/tools/web-parse",           {"url":"https://example.com"}),
    ("GET",  "/v1/memory/test_user",          None),
    ("POST", "/v1/memory/test_user/search",   {"query":"test","top_k":3}),
    ("POST", "/v1/image/generate",            {"prompt":"cat"}),
    ("POST", "/v1/vlm/analyze",               {"image_url":"https://example.com/img.jpg","question":"что на фото?"}),
    ("POST", "/v1/tools/generate-pptx",       {"topic":"AI","slides_count":3,"user_id":"u1"}),
]

print(f"{'Method':<6} {'Endpoint':<35} {'Status':>6}  Result")
print("-" * 70)
for method, path, body in endpoints:
    try:
        if method == "GET":
            r = httpx.get(f"{BASE}{path}", timeout=10)
        else:
            r = httpx.post(f"{BASE}{path}", json=body, timeout=15)
        icon = "✅" if r.status_code < 400 else "⚠️ "
        print(f"{method:<6} {path:<35} {r.status_code:>6}  {icon}")
    except Exception as e:
        print(f"{method:<6} {path:<35} {'ERR':>6}  ❌ {str(e)[:40]}")